In [44]:
import itertools
import multiprocessing
import multiprocessing.pool
import os
import warnings

import alignparse
import alignparse.targets

import dms_variants.codonvarianttable
from dms_variants.constants import CBPALETTE
import dms_variants.illuminabarcodeparser
import dms_variants.utils
import dms_variants.plotnine_themes

from IPython.display import display, HTML

import pandas as pd

from plotnine import *

import yaml

In [45]:
theme_set(dms_variants.plotnine_themes.theme_graygrid())

In [46]:
print(f"Using alignparse version {alignparse.__version__}")
print(f"Using dms_variants version {dms_variants.__version__}")

Using alignparse version 0.6.2
Using dms_variants version 1.6.0


In [47]:
warnings.simplefilter('ignore')

In [48]:
with open('config.yaml') as f:
    config = yaml.safe_load(f)

In [49]:
os.makedirs(config['counts_dir'], exist_ok=True)
os.makedirs(config['figs_dir'], exist_ok=True)

In [50]:
targets = alignparse.targets.Targets(seqsfile=config['amplicons_aLIHA'],
                                     feature_parse_specs=config['feature_parse_specs_aLIHA'])
geneseq = targets.get_target(config['primary_target']).get_feature('gene').seq
print(f"Read gene of {len(geneseq)} nt for {config['primary_target']} from {config['amplicons_aLIHA']}")
      
print('Initializing CodonVariantTable from gene sequence and ' +
      config['codon_variant_table_file_aLIHA'])
      
variants = dms_variants.codonvarianttable.CodonVariantTable(
                geneseq=geneseq,
                barcode_variant_file=config['codon_variant_table_file_aLIHA'],
                substitutions_are_codon=True,
                substitutions_col='codon_substitutions',
                primary_target=config['primary_target'])

Read gene of 540 nt for aLIHA from data/PacBio_amplicon_aLIHA.gb
Initializing CodonVariantTable from gene sequence and results/variants/codon_variant_table_aLIHA.csv


In [51]:
variants = pd.read_csv(config['codon_variant_table_file_aLIHA'], na_filter=None)

variants = variants.reset_index(drop=True)

display(HTML(variants.head().to_html(index=False)))

target,library,barcode,variant_call_support,codon_substitutions,aa_substitutions,n_codon_substitutions,n_aa_substitutions
aLIHA,pool1,AAAAAAAAAGAACTAG,25,AAG153TGT,K153C,1,1
aLIHA,pool1,AAAAAAAAGATCTAGG,8,CAC137CAA,H137Q,1,1
aLIHA,pool1,AAAAAAACCAACCCTT,33,ACC90ATT,T90I,1,1
aLIHA,pool1,AAAAAAACCCCAAAAT,11,GGC83ATT,G83I,1,1
aLIHA,pool1,AAAAAAAGCAGGACCC,5,ACA111TTG,T111L,1,1


In [52]:
dup_barcodes = (
    variants
    .groupby(['library', 'barcode'])
    .size()
    .rename('duplicate_count')
    .reset_index()
    .query('duplicate_count > 1')
    )

print('Here are duplicated barcodes:')
display(HTML(dup_barcodes.head().to_html(index=False)))

print(f"\nRemoving the {len(dup_barcodes)} duplicated barcodes."
      f"Started with {len(variants)} barcodes:")
variants = (
    variants
    .merge(dup_barcodes, on=['library', 'barcode'], how='outer')
    .query('duplicate_count.isnull()', engine='python')
    )
print(f"After removing duplicates, there are {len(variants)} barcodes.")

Here are duplicated barcodes:


library,barcode,duplicate_count



Removing the 0 duplicated barcodes.Started with 133834 barcodes:
After removing duplicates, there are 133834 barcodes.


In [53]:
targets = alignparse.targets.Targets(seqsfile=config['amplicons_aLIHA'],
                                     feature_parse_specs=config['feature_parse_specs_aLIHA'])

In [54]:
barcode_runs = (pd.read_csv(config['barcode_runs'])
                 .assign(R1=lambda x: x['R1'].str.split('; '))
                 )

display(HTML(barcode_runs.to_html(index=False)))

library,sample,sample_type,sort_bin,concentration,date,number_cells,R1
pool1,TiteSeq_01_bin1,TiteSeq,1,1.0,250106,3582123,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s1_b1_p1_seq1_S1_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s1_b1_p1_seq2_S19_L003_R1_001.fastq.gz]"
pool1,TiteSeq_01_bin2,TiteSeq,2,1.0,250106,1417040,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s1_b2_p1_seq1_S2_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s1_b2_p1_seq2_S20_L003_R1_001.fastq.gz]"
pool1,TiteSeq_01_bin3,TiteSeq,3,1.0,250106,425422,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s1_b3_p1_seq1_S3_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s1_b3_p1_seq2_S21_L003_R1_001.fastq.gz]"
pool1,TiteSeq_01_bin4,TiteSeq,4,1.0,250106,1994308,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s1_b4_p1_seq1_S4_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s1_b4_p1_seq2_S22_L003_R1_001.fastq.gz]"
pool1,TiteSeq_02_bin1,TiteSeq,1,2.0,250106,3462651,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s2_b1_p1_seq1_S5_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s2_b1_p1_seq2_S23_L003_R1_001.fastq.gz]"
pool1,TiteSeq_02_bin2,TiteSeq,2,2.0,250106,1424183,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s2_b2_p1_seq1_S6_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s2_b2_p1_seq2_S24_L003_R1_001.fastq.gz]"
pool1,TiteSeq_02_bin3,TiteSeq,3,2.0,250106,446906,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s2_b3_p1_seq1_S7_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s2_b3_p1_seq2_S25_L003_R1_001.fastq.gz]"
pool1,TiteSeq_02_bin4,TiteSeq,4,2.0,250106,1826652,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s2_b4_p1_seq1_S8_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s2_b4_p1_seq2_S26_L003_R1_001.fastq.gz]"
pool1,TiteSeq_03_bin1,TiteSeq,1,3.0,250106,4121017,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s3_b1_p1_seq1_S9_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s3_b1_p1_seq2_S27_L003_R1_001.fastq.gz]"
pool1,TiteSeq_03_bin2,TiteSeq,2,3.0,250106,1242734,"[/mnt/campbell-data/jhollis/20250106_LH00740_0036_B22W57FLT3/Unaligned/LFA_P1/s3_b2_p1_seq1_S10_L001_R1_001.fastq.gz, /mnt/campbell-data/jhollis/20250114_LH00740_0038_B22W5C5LT3/Unaligned/LFA1_P2/s3_b2_p1_seq2_S28_L003_R1_001.fastq.gz]"


In [55]:
assert len(barcode_runs) == len(barcode_runs.groupby(['library', 'sample']))

In [56]:
unknown_libs = set(barcode_runs['library']) - set(variants['library'])
if unknown_libs:
    raise ValueError(f"Libraries with barcode runs not in variant table: {unknown_libs}")

In [57]:
bclen = len(targets.targets[0].get_feature('barcode').seq)

assert (bclen == len(target.get_feature('barcode').seq) for target in targets.targets)

print(f"Barcodes of length {bclen}")

Barcodes of length 16


In [58]:
parser_params = config['illumina_barcode_parser_params']

display(HTML(
    pd.Series(parser_params, name='value')
    .rename_axis(index='parameter')
    .reset_index()
    .to_html(index=False)
    ))

parameter,value
upstream,GGCCGC
downstream,
minq,20
upstream_mismatch,1
downstream_mismatch,0


In [59]:
parsers = {lib: dms_variants.illuminabarcodeparser.IlluminaBarcodeParser(
                    bclen=bclen,
                    valid_barcodes=variants.loc[variants['library']==lib]['barcode'],
                    **parser_params)
           for lib in set(variants['library'])}

print('Number of valid barcodes searched for by each parser:')
display(HTML(
    pd.DataFrame([(lib, len(p.valid_barcodes)) for lib, p in parsers.items()],
                 columns=['library', 'number of valid barcodes'])
    .to_html(index=False)
    ))

Number of valid barcodes searched for by each parser:


library,number of valid barcodes
pool1,76345
pool2,57489


In [60]:
ncpus = min(config['max_cpus'], multiprocessing.cpu_count())
print(f"Using {ncpus} CPUs")

Using 8 CPUs


In [63]:
def process_func(parser, r1files, library, sample):
    """Convenience function to be starmapped to multiprocessing pool."""
    return parser.parse(r1files, add_cols={'library': library, 'sample': sample})

# parallel computation of list of data frames
with multiprocessing.pool.Pool(processes=ncpus) as pool:
    bclist = pool.starmap(
                process_func,
                [(parsers[run.library], run.R1, run.library, run.sample)
                  for run in barcode_runs.itertuples()],
                )

AttributeError: 'Pandas' object has no attribute 'R2'

In [62]:
counts = pd.concat([samplecounts for samplecounts, _ in bclist],
                   sort=False,
                   ignore_index=True)

print('First few lines of counts data frame:')
display(HTML(counts.head().to_html(index=False)))

fates = pd.concat([samplefates for _, samplefates in bclist],
                  sort=False,
                  ignore_index=True)

print('First few lines of fates data frame:')
display(HTML(fates.head().to_html(index=False)))

First few lines of counts data frame:


barcode,count,library,sample
ATTTAATGCTACGCCC,106447,pool1,SortSeq_bin1
ACTTAACCCCCACCTT,60952,pool1,SortSeq_bin1
GTTCCCTACCGCTTCA,53576,pool1,SortSeq_bin1
CCACCGAACTTCCATA,52976,pool1,SortSeq_bin1
CCGGCGGCGTGCCCCC,43310,pool1,SortSeq_bin1


First few lines of fates data frame:


fate,count,library,sample
invalid barcode,95805815,pool1,SortSeq_bin1
valid barcode,31967548,pool1,SortSeq_bin1
low quality barcode,7166721,pool1,SortSeq_bin1
unparseable barcode,2948418,pool1,SortSeq_bin1
failed chastity filter,0,pool1,SortSeq_bin1


In [20]:
fatesfile = os.path.join(config['counts_dir'], 'barcode_fates.csv')
print(f"Writing barcode fates to {fatesfile}")
fates.to_csv(fatesfile, index=False)

Writing barcode fates to results/counts/barcode_fates.csv


In [21]:
display(HTML(fates
             .pivot_table(columns='fate',
                          values='count',
                          index=['library', 'sample'])
             .to_html()
             ))

In [43]:
barcode_fate_plot = (
    ggplot(
        fates
        .assign(sample=lambda x: pd.Categorical(x['sample'],
                                                x['sample'].unique(),
                                                ordered=True),
                fate=lambda x: pd.Categorical(x['fate'],
                                              x['fate'].unique(),
                                              ordered=True),
                is_valid=lambda x: x['fate'] == 'valid barcode'
                ), 
        aes('fate', 'count', fill='is_valid')) +
    geom_bar(stat='identity') +
    facet_grid('sample ~ library') +
    facet_grid('sample ~ library') +
    scale_fill_manual(CBPALETTE, guide=False) +
    theme(figure_size=(1 * (1 + fates['library'].nunique()),
                       1 * (1.2 + fates['sample'].nunique())),
          axis_text_x=element_text(angle=90),
          panel_grid_major_x=element_blank()
          ) +
    scale_y_continuous(labels=dms_variants.utils.latex_sci_not,
                       name='number of reads')
    )

_ = barcode_fate_plot.draw()

In [34]:
variants.add_sample_counts_df(counts)

AttributeError: 'DataFrame' object has no attribute 'add_sample_counts_df'

In [35]:
print(f"Writing variant counts to {config['variant_counts_file']}")
counts.to_csv(config['variant_counts_file'], index=False)

Writing variant counts to results/counts/variant_counts.csv


In [25]:
barcode_fate_plot = (
    ggplot(
        fates
        .assign(sample=lambda x: pd.Categorical(x['sample'],
                                                x['sample'].unique(),
                                                ordered=True),
                fate=lambda x: pd.Categorical(x['fate'],
                                              x['fate'].unique(),
                                              ordered=True),
                is_valid=lambda x: x['fate'] == 'valid barcode'
                ), 
        aes('fate', 'count', fill='is_valid')) +
    geom_bar(stat='identity') +
    facet_grid('sample ~ library') +
    facet_grid('sample ~ library') +
    scale_fill_manual(CBPALETTE, guide=False) +
    theme(figure_size=(7 * (1 + fates['library'].nunique()),
                       7 * (1.2 + fates['sample'].nunique())),
          axis_text_x=element_text(angle=90),
          panel_grid_major_x=element_blank()
          ) +
    scale_y_continuous(labels=dms_variants.utils.latex_sci_not,
                       name='number of reads')
    )

_ = barcode_fate_plot.draw()